In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

WORKING = Path('data/working_tmdb')

def load(name):
    with open(WORKING / f'{name}.pkl', 'rb') as f:
        return pickle.load(f)

embeddings = load('embeddings')   # [N, 384] float32, L2-normalized
metadata   = load('metadata')     # DataFrame

print('=' * 60)
print('TMDB SIMILAR MOVIES — PHASE 2')
print('=' * 60)
print(f'  Embeddings shape : {embeddings.shape}')
print(f'  Metadata rows    : {len(metadata):,}')
print(f'  Year range       : {metadata["year"].min()} – {metadata["year"].max()}')


TMDB SIMILAR MOVIES — PHASE 2
  Embeddings shape : (4803, 384)
  Metadata rows    : 4,803
  Year range       : 0 – 2017


In [2]:
# Core Similarity Function 
import numpy as np
import pandas as pd

def find_similar(query: str, top_n: int = 20, year_from: int = None,
                 year_to: int = None) -> dict:

    q_lower = query.lower().strip()

    # ── Find the query movie 
    exact    = metadata[metadata['title'].str.lower() == q_lower]
    starts   = metadata[metadata['title'].str.lower().str.startswith(q_lower)]
    contains = metadata[metadata['title'].str.lower().str.contains(
        q_lower, regex=False, na=False)]

    candidates = pd.concat([exact, starts, contains]).drop_duplicates('movie_id')

    if candidates.empty:
        return {'error': f'No movie found matching "{query}"'}

    query_row = candidates.iloc[0]
    query_idx = int(query_row['row_idx'])
    query_vec = embeddings[[query_idx]]    

    # ── Cosine similarity against all movies 
  
    sims = (embeddings @ query_vec.T).squeeze() 
    sims[query_idx] = -1.0                         

    # year filter 
    # Build a mask: 1.0 for movies in the year range, -2.0 for those outside
    # (this effectively excludes out-of-range movies while keeping the
    # similarity values intact for in-range ones)
    if year_from is not None or year_to is not None:
        years = metadata['year'].values
        mask  = np.ones(len(sims), dtype=np.float32)
        if year_from is not None:
            mask[years < year_from] = 0.0
        if year_to is not None:
            mask[years > year_to]   = 0.0
        sims = sims * mask - 2.0 * (1.0 - mask)   

    # Top-N 
    top_idxs = sims.argsort()[::-1][:top_n]

    results = []
    for idx in top_idxs:
        row = metadata.iloc[idx]
        results.append({
            'movie_id'  : int(row['movie_id']),
            'title'     : str(row['title']),
            'genres'    : str(row['genres']),
            'year'      : int(row['year']) if row['year'] > 0 else None,
            'similarity': round(float(sims[idx]), 4),
        })

    return {
        'query': {
            'movie_id': int(query_row['movie_id']),
            'title'   : str(query_row['title']),
            'genres'  : str(query_row['genres']),
            'year'    : int(query_row['year']) if query_row['year'] > 0 else None,
        },
        'similar': results,
    }

print('find_similar() defined.')


find_similar() defined.


In [4]:
# Test the Similarity Function 
print('=' * 60)
print('TESTING SIMILARITY FUNCTION')
print('=' * 60)

test_cases = [
    ('The Dark Knight',      None, None),
    ('Toy Story',            None, None),
    ('Inception',            None, None),
    ('The Matrix',           2000, None),  
    ('Interstellar',         None, None),
]

for title, y_from, y_to in test_cases:
    result = find_similar(title, top_n=5, year_from=y_from, year_to=y_to)
    if 'error' in result:
        print(f'  ERROR: {result["error"]}')
        continue
    q = result['query']
    print(f'\n  Similar to "{q["title"]}" ({q["year"]}) — {q["genres"]}')
    yr_note = f' (year >= {y_from})' if y_from else ''
    print(f'  {yr_note}')
    print(f'  {"Sim":>6}  Title')
    print('  ' + '-' * 48)
    for r in result['similar']:
        yr = f'({r["year"]})' if r['year'] else ''
        print(f'  {r["similarity"]:>6.4f}  {r["title"]} {yr}')


TESTING SIMILARITY FUNCTION

  Similar to "The Dark Knight" (2008) — Drama|Action|Crime|Thriller
  
     Sim  Title
  ------------------------------------------------
  0.8587  The Dark Knight Rises (2012)
  0.7697  Batman (1989)
  0.7387  Batman v Superman: Dawn of Justice (2016)
  0.7382  Batman Begins (2005)
  0.7313  Batman Forever (1995)

  Similar to "Toy Story" (1995) — Animation|Comedy|Family
  
     Sim  Title
  ------------------------------------------------
  0.8051  Toy Story 2 (1999)
  0.7336  Toy Story 3 (2010)
  0.5240  Big (1988)
  0.5127  St. Vincent (2014)
  0.5094  Ted 2 (2015)

  Similar to "Inception" (2010) — Action|Thriller|Science Fiction|Mystery|Adventure
  
     Sim  Title
  ------------------------------------------------
  0.5720  Memento (2000)
  0.5601  Kiss Kiss Bang Bang (2005)
  0.5543  The Spanish Prisoner (1997)
  0.5520  Identity Thief (2013)
  0.5479  Hearts in Atlantis (2001)

  Similar to "The Matrix" (1999) — Action|Science Fiction
   (year >= 2

In [5]:
# Search Function for the UI 


def search_movies_tmdb(q: str, limit: int = 15) -> list:
    """
    Search TMDB movies by title substring.
    Starts-with results come first, then contains.
    Returns list of dicts: [{'movie_id', 'title', 'year', 'genres'}, ...]
    """
    q_lower = q.lower().strip()
    if not q_lower:
        return []

    starts   = metadata[metadata['title'].str.lower().str.startswith(q_lower)]
    contains = metadata[
        ~metadata['title'].str.lower().str.startswith(q_lower) &
        metadata['title'].str.lower().str.contains(q_lower, regex=False, na=False)
    ]
    combined = pd.concat([starts, contains]).head(limit)

    results = []
    seen    = set()
    for _, row in combined.iterrows():
        mid = int(row['movie_id'])
        if mid in seen:
            continue
        seen.add(mid)
        results.append({
            'movie_id': mid,
            'title'   : str(row['title']),
            'year'    : int(row['year']) if row['year'] > 0 else None,
            'genres'  : str(row['genres']),
        })
    return results


print('search_movies_tmdb("inter") →')
for r in search_movies_tmdb('inter', limit=5):
    print(f'  {r["title"]} ({r["year"]})')

print()
print('search_movies_tmdb("the dark") →')
for r in search_movies_tmdb('the dark', limit=5):
    print(f'  {r["title"]} ({r["year"]})')


search_movies_tmdb("inter") →
  Interstellar (2014)
  Interview with the Vampire (1994)
  Interview with the Assassin (2002)
  Captain America: The Winter Soldier (2014)
  The Huntsman: Winter's War (2016)

search_movies_tmdb("the dark") →
  The Dark Knight Rises (2012)
  The Dark Knight (2008)
  The Darkest Hour (2011)
  The Dark Hours (2005)
  Thor: The Dark World (2013)


In [6]:
# Popular Movies List for UI 


def get_popular_tmdb(n: int = 50) -> list:
    """Return n popular TMDB movies sorted by vote_count."""
    top = (metadata.sort_values('rating_count', ascending=False)
                   .head(n)
                   .reset_index(drop=True))
    results = []
    for _, row in top.iterrows():
        results.append({
            'movie_id': int(row['movie_id']),
            'title'   : str(row['title']),
            'year'    : int(row['year']) if row['year'] > 0 else None,
            'genres'  : str(row['genres']),
        })
    return results

print(f'Top 10 popular TMDB movies:')
for r in get_popular_tmdb(10):
    print(f'  {r["title"]} ({r["year"]})  —  {r["genres"][:40]}')


Top 10 popular TMDB movies:
  Inception (2010)  —  Action|Thriller|Science Fiction|Mystery|
  The Dark Knight (2008)  —  Drama|Action|Crime|Thriller
  Avatar (2009)  —  Action|Adventure|Fantasy|Science Fiction
  The Avengers (2012)  —  Science Fiction|Action|Adventure
  Deadpool (2016)  —  Action|Adventure|Comedy
  Interstellar (2014)  —  Adventure|Drama|Science Fiction
  Django Unchained (2012)  —  Drama|Western
  Guardians of the Galaxy (2014)  —  Action|Science Fiction|Adventure
  The Hunger Games (2012)  —  Science Fiction|Adventure|Fantasy
  Mad Max: Fury Road (2015)  —  Action|Adventure|Science Fiction|Thrille


In [8]:
# Save Inference Bundle 
import pickle
from pathlib import Path

Path('data/inference_tmdb').mkdir(parents=True, exist_ok=True)

bundle = {
    # The full normalized embedding matrix — cosine sim is just a dot product
    'embeddings'  : embeddings,     # np.ndarray [N, 384] float32
    # Movie metadata for display and search
    'metadata'    : metadata,       # pd.DataFrame
    # Embedding dimension info
    'emb_dim'     : embeddings.shape[1],   # 384
    'num_movies'  : len(metadata),
}

out_path = 'data/inference_tmdb/inference_bundle_tmdb.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(bundle, f, protocol=4)

size_mb = Path(out_path).stat().st_size / 1e6
print(f'Saved: {out_path}  ({size_mb:.1f} MB)')
print()
print('Bundle contents:')
for key, val in bundle.items():
    if hasattr(val, 'shape') and hasattr(val, 'dtype'):
        desc = f'numpy {val.shape} {val.dtype}'
    elif hasattr(val, '__len__') and not isinstance(val, str):
        desc = f'{type(val).__name__}  {len(val):,} items'
    else:
        desc = str(val)
    print(f'  {key:15s}: {desc}')
print()


Saved: data/inference_tmdb/inference_bundle_tmdb.pkl  (9.3 MB)

Bundle contents:
  embeddings     : numpy (4803, 384) float32
  metadata       : DataFrame  4,803 items
  emb_dim        : 384
  num_movies     : 4803

